# Text2Cypher Agent

This notebook builds an autonomous agent that discovers a graph schema and writes its own Cypher queries — the **Text2Cypher** retrieval pattern. The agent uses the **Strands Agents SDK** with the **Model Context Protocol (MCP)** to connect to Neo4j.

The previous notebook used pre-written Cypher templates in `@tool` functions — reliable but limited to predefined query patterns. This notebook takes the opposite approach: the agent retrieves the schema, reasons about the question, and generates original Cypher from scratch. This is more flexible but requires a schema-first approach to produce accurate queries.

Strands takes a model-driven approach: the model decides what to do at each step rather than following a developer-defined workflow. Its native MCP support means the agent discovers Neo4j's tools at runtime and calls them directly.

> **Note:** LangGraph is a viable alternative framework for this pattern. It provides fine-grained control over the agent loop and is better suited for complex, multi-step workflows. This workshop uses Strands for its simpler API and built-in MCP support.

**What you'll do:**
- Connect to a Neo4j MCP Server through an AWS AgentCore Gateway
- Let the agent discover the graph schema via MCP tool discovery
- Ask natural language questions and watch the agent write its own Cypher
- Compare this Text2Cypher approach to the previous notebook's pre-written Cypher templates

**Prerequisites:**
- `../CONFIG.txt` configured with `MCP_GATEWAY_URL` and `MCP_ACCESS_TOKEN`

## 1. Setup

Install the required packages for the Strands MCP agent.

In [ ]:
%pip install strands-agents strands-agents-tools mcp httpx -q

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

print("All imports successful.")

## 2. Configuration

Load settings from `CONFIG.txt`. The MCP Gateway URL and access token authenticate your connection to the Neo4j MCP Server running on AWS.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv("../CONFIG.txt")

MODEL_ID = os.getenv("MODEL_ID")
REGION = os.getenv("REGION", "us-east-1")
MCP_GATEWAY_URL = os.getenv("MCP_GATEWAY_URL")
MCP_ACCESS_TOKEN = os.getenv("MCP_ACCESS_TOKEN")

assert MCP_GATEWAY_URL, "MCP_GATEWAY_URL not configured in CONFIG.txt"
assert MCP_ACCESS_TOKEN, "MCP_ACCESS_TOKEN not configured in CONFIG.txt"

print(f"Model:     {MODEL_ID}")
print(f"Region:    {REGION}")
print("\nConfiguration loaded!")

## 3. Initialize Model & MCP Connection

- **BedrockModel**: Configures Claude on Amazon Bedrock with temperature 0 for deterministic responses
- **MCPClient**: Connects to the Neo4j MCP Server and discovers available tools. `list_tools_sync()` returns tool wrappers that the agent can call directly — this is the standard Strands pattern for MCP integration.

In [ ]:
from botocore.config import Config as BotocoreConfig

bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    temperature=0,
    boto_client_config=BotocoreConfig(read_timeout=300),
)

# Open a persistent MCP connection for the notebook session
mcp_client = MCPClient(lambda: streamablehttp_client(
    url=MCP_GATEWAY_URL,
    headers={"Authorization": f"Bearer {MCP_ACCESS_TOKEN}"},
))
mcp_client.__enter__()

# Discover available tools from the MCP server
mcp_tools = mcp_client.list_tools_sync()
tool_names = [t.tool_name for t in mcp_tools]
print(f"MCP tools discovered: {tool_names}")
print(f"\nModel: {MODEL_ID}")

## 4. Create Agent & Query Function

Define the system prompt and create the agent with the discovered MCP tools. The `query()` function sends questions to the agent and prints the response.

Unlike the previous notebook's pre-written `@tool` Cypher templates, this agent writes its own Cypher from scratch — the **Text2Cypher** pattern.

In [ ]:
SYSTEM_PROMPT = """You are a Neo4j database assistant with access to a knowledge graph 
containing SEC 10-K financial filing data (companies, products, services, risk factors, 
financial metrics, executives, asset manager holdings).

Rules:
1. Always retrieve the database schema before writing Cypher queries.
2. Only use read-only Cypher (MATCH, RETURN, WITH, WHERE, ORDER BY, LIMIT).
3. Include LIMIT clauses to avoid excessive results.
4. Use COALESCE() or IS NOT NULL for properties that might be missing.
5. Format results clearly and cite actual data from query results.
6. Modern Cypher syntax:
   - Use elementId(n) instead of id(n) — id() is removed in Neo4j 5+
   - Use COUNT{ pattern } instead of size((pattern)) for counting pattern occurrences
   - Use EXISTS{ pattern } instead of exists((pattern)) for checking pattern existence
   - Always use $parameter syntax for dynamic values, never string concatenation
"""

agent = Agent(
    model=bedrock_model,
    system_prompt=SYSTEM_PROMPT,
    tools=mcp_tools,
)


def query(question: str):
    """Ask the agent a question about the knowledge graph."""
    print(f'Question: "{question}"')
    print("-" * 60)
    response = agent(question)
    print(f"\n{response}")
    return response


print("Agent ready!")

## 5. Demo Queries

Run the following queries to see the agent discover the schema and query the graph. Watch the tool calls in the output to understand how the agent reasons about each question.

In [ ]:
query("What is the database schema?")

In [ ]:
query("How many nodes are there by label?")

In [ ]:
query("Show 5 sample records from the most populated node type.")

## 6. Your Query

Try your own questions about the SEC financial data:

- "How many companies are in the database?"
- "What products does Apple offer?"
- "Which asset managers own stakes in NVIDIA?"
- "What risk factors does Microsoft face?"
- "Show me the financial metrics for Tesla."
- "Who are the executives at Amazon?"
- "Which companies face risk factors related to cybersecurity?"

In [ ]:
# query("Your question here")

## Summary

You built an autonomous Text2Cypher agent that writes its own Cypher queries from scratch:

| Component | Role |
|-----------|------|
| **Strands Agent** | Receives your question, retrieves the schema, writes Cypher, executes it, interprets results |
| **MCPClient** | Discovers tools from the MCP server and provides them to the agent |
| **Neo4j MCP Server** | Exposes `get-schema` and `read-cypher` tools over MCP |
| **BedrockModel** | Powers the agent's reasoning with Claude on Amazon Bedrock |

### Cypher Templates vs Text2Cypher

| Aspect | Notebook 02 (Cypher Templates) | This Notebook (Text2Cypher) |
|--------|-------------------------------|----------------------------|
| **Cypher source** | Pre-written in `@tool` functions | Agent writes from scratch after schema discovery |
| **Flexibility** | Limited to predefined query patterns | Any question the schema can answer |
| **Reliability** | High — expert-reviewed queries | Variable — depends on LLM reasoning quality |

The schema-first approach is critical for Text2Cypher: by retrieving the graph structure before writing queries, the agent uses correct node labels, relationship types, and property names rather than guessing.

---

This completes Lab 5. You progressed from MCP tool discovery (notebook 01) to pre-written Cypher templates (notebook 02) to fully autonomous query generation (this notebook) — three points on the spectrum from controlled to autonomous retrieval.

In [ ]:
# mcp_client.__exit__(None, None, None)